In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.preprocessing import MinMaxScaler


In [2]:
elo = pd.read_csv("combined_elo_usernames.csv")
# elo.drop(columns=["username"], inplace=True)
tournament_score = pd.read_csv("combined_player_weighted_stats.csv")
#check for duplicated user_id in both dataframes
print(elo[elo.duplicated(subset=['user_id'], keep=False)])
print(tournament_score[tournament_score.duplicated(subset=['user_id'], keep=False)])
merged_df = pd.merge(elo, tournament_score, on="user_id", how="outer")
merged_df.drop(columns=["median_pScore", "total_maps_played"], inplace=True)
merged_df['username'] = merged_df.apply(
    lambda row: row['username'] if pd.notna(row['username']) else row['Player Name'], axis=1
)
merged_df.drop(columns=["Player Name"], inplace=True)
merged_df = merged_df[~merged_df['username'].isin(['Banned', 'Unavailable'])]
merged_df.fillna(0, inplace=True)

# # save merged_df as csv
# merged_df.to_csv("merged.csv", index=False)

Empty DataFrame
Columns: [user_id, elo, username, country]
Index: []
Empty DataFrame
Columns: [user_id, Player Name, median_pScore, total_maps_played, num_appearances, adjusted_median_pScore]
Index: []


In [3]:


filtered_df = merged_df[(merged_df['adjusted_median_pScore'] != 0) & (merged_df['num_appearances'] > 0) & (merged_df['num_appearances'] < 7) & (merged_df['elo'] != 0)]

filtered_df['elo'] = pd.to_numeric(filtered_df['elo'], errors='coerce')
filtered_df['adjusted_median_pScore'] = pd.to_numeric(filtered_df['adjusted_median_pScore'], errors='coerce')

C:\Users\Acer\AppData\Local\Temp\ipykernel_3096\1062281366.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['elo'] = pd.to_numeric(filtered_df['elo'], errors='coerce')
C:\Users\Acer\AppData\Local\Temp\ipykernel_3096\1062281366.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['adjusted_median_pScore'] = pd.to_numeric(filtered_df['adjusted_median_pScore'], errors='coerce')


In [ ]:
from math import sqrt
import numpy as np

mania_4k_rankings = pd.read_csv("mania_4k_rankings.csv")

# Filter players with no tournament score
# no_score_players = merged_df[merged_df['adjusted_median_pScore'] == 0].copy()
no_score_players = merged_df.copy()

# Prepare player_data list
player_data = []
for _, row in no_score_players.iterrows():
    player_data.append({
        'player_id': row['user_id'],
        'rank_4k': mania_4k_rankings.loc[mania_4k_rankings['user_id'] == row['user_id'], 'global_rank'].values[0] if not mania_4k_rankings.loc[mania_4k_rankings['user_id'] == row['user_id'], 'global_rank'].isna().all() else np.nan,
        'elo_score': row['elo'] if row['elo'] != 'No Elo' else np.nan,
    })

# Remove players missing rank_4k or elo_score
player_data = [p for p in player_data if not np.isnan(p['rank_4k']) and not np.isnan(p['elo_score'])]

if player_data:
    max_rank = max(p['rank_4k'] for p in player_data)
    max_elo = max(p['elo_score'] for p in player_data)
    min_elo = min(p['elo_score'] for p in player_data)

    # Step 1: Normalize rank and elo
    for p in player_data:
        p['norm_rank'] = 1 - (p['rank_4k'] / max_rank)
        p['norm_elo'] = (p['elo_score'] - min_elo) / (max_elo - min_elo)

    # Step 2: Overperformance
    for p in player_data:
        p['overperformance'] = p['norm_elo'] - p['norm_rank']

    # Step 3: Normalize overperformance
    over_list = [p['overperformance'] for p in player_data]
    max_over = max(over_list)
    min_over = min(over_list)
    for p in player_data:
        p['norm_overperformance'] = (p['overperformance'] - min_over) / (max_over - min_over) if max_over != min_over else 0

    # Step 4: Z-score
    mean_over = np.mean(over_list)
    std_dev = np.std(over_list, ddof=1)
    for p in player_data:
        p['z_score'] = (p['overperformance'] - mean_over) / std_dev if std_dev != 0 else 0


    # Step 5: Adjusted weight
    for p in player_data:
        if p['z_score'] > 1.96:
            p['adjusted_weight'] = 0.3
        elif p['z_score'] < -1.96:
            p['adjusted_weight'] = 0.07
        else:
            p['adjusted_weight'] = 0.0

    # Step 6: Combined score
    for p in player_data:
        p['combined_score'] = (
            0.8 * p['norm_rank'] +
            0.2 * p['norm_elo'] +
            p['norm_overperformance'] * p['adjusted_weight']
        )

    # Step 7: Adjusted rank
    sorted_players = sorted(player_data, key=lambda x: x['combined_score'], reverse=True)
    for idx, p in enumerate(sorted_players, 1):
        p['adjusted_rank'] = idx

    # Step 9: Handle ties & missing data
    # Sort by final_rank, then rank_4k for ties
    sorted_players = sorted(sorted_players, key=lambda x: (x['adjusted_rank'], x['rank_4k']))

    # Output
    output = []
    for p in sorted_players:
        output.append({
            'player_id': p['player_id'],
            'elo_score': p['elo_score'],
            'rank_4k': p['rank_4k'],
            'final_rank': p['adjusted_rank'],
            'country': merged_df.loc[merged_df['user_id'] == p['player_id'], 'country'].values[0] if not merged_df.loc[merged_df['user_id'] == p['player_id'], 'country'].isna().all() else 'Unknown',
            'username': merged_df.loc[merged_df['user_id'] == p['player_id'], 'username'].values[0] if not merged_df.loc[merged_df['user_id'] == p['player_id'], 'username'].isna().all() else 'Unknown'
        })
else:
    output = []

# Transform final_rank to 1/final_rank
output_df = pd.DataFrame(output)
print(output_df)


# Calculate percentile rank for final_rank (1 for top, 0 for bottom)
output_df['final_rank_percentile'] = output_df['final_rank'].rank(method='max', ascending=False, pct=True)
output_df.to_csv("leaderboard_rank_adjusted.csv", index=False)
